# Group 11: Person B experiments (Q3 unlabeled data, Q5 robustness)

**Kaggle:** open this notebook from the competition page (or click *Add Input* → the competition), switch **Internet on** in the right-hand panel, then **Run All**. Nothing else to set up: the data is read from `/kaggle/input` and every result is saved to the *Output* tab.

**Colab** also works: add `KAGGLE_USERNAME` / `KAGGLE_KEY` under 🔑 Secrets once, then Run all.

Runtime is about 10–15 minutes on CPU.

In [ ]:
import os, shutil, subprocess, sys
REPO_URL = "https://github.com/hisylvia66/human-or-ai-submissions.git"
BRANCH   = "submission/group-number-11"
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content" if os.path.exists("/content") else os.getcwd()
REPO = os.path.join(WORK, "repo")
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(os.path.join(REPO, "submissions/group-number-11"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("working in", os.getcwd())

In [ ]:
from data_utils import ensure_data
DATA = ensure_data("data")   # /kaggle/input/... on Kaggle; downloaded on Colab
print(DATA, sorted(os.listdir(DATA)))

## Q3: learning from unlabeled data

In [ ]:
!python run_unlabeled_experiment.py --data_dir data

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 250, "display.max_columns", 30)
for f in ["unlabeled_experiment_results.csv", "unlabeled_pseudo_label_noise.csv",
          "unlabeled_domain_shift.csv", "unlabeled_confidence_by_set.csv"]:
    print(f); display(pd.read_csv(f).round(4))
display(Image("unlabeled_self_training.png"))

## Q5: robustness improvement (also writes `submission_R2.csv`, trained on all 10k labeled rows)

In [ ]:
!python run_robustness_improvement.py --data_dir data --submit R2

In [ ]:
for f in ["robustness_improvement_results.csv", "robustness_char_census.csv"]:
    print(f); display(pd.read_csv(f).round(4))

## Save everything to the output folder

In [ ]:
OUT = os.path.join(WORK, "personB_results"); os.makedirs(OUT, exist_ok=True)
files = ["unlabeled_experiment_results.csv", "unlabeled_pseudo_label_noise.csv", "unlabeled_domain_shift.csv",
         "unlabeled_confidence_by_set.csv", "unlabeled_self_training.png",
         "robustness_improvement_results.csv", "robustness_char_census.csv",
         "robustness_validation_scores.csv", "submission_R2.csv"]
for f in files:
    if os.path.exists(f): shutil.copy(f, OUT)
shutil.make_archive(OUT, "zip", OUT)
print("saved:", sorted(os.listdir(OUT)), "\nzip:", OUT + ".zip")
if WORK == "/content":
    from google.colab import files as colab_files; colab_files.download(OUT + ".zip")